In [14]:
# the following is because I am using a local version of pypsa2smspp and pySMSpp
# which allows me to git pull easily (without recreating my conda env)
%load_ext autoreload
%autoreload 2

import os
import pypsa
import netCDF4 as nc
import pandas as pd
import numpy as np
import math
#import re
import CreateData
from CreateData import *
import pypsa2smspp
import pysmspp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# read configuration file and non time series data
cfg=read_plan4res_config(config_path="settings/",settings="settings_power_uc_stoch.yml")
Coupling=read_coupling(cfg)
ZP=read_ZP(cfg,Coupling) # defines the regions
Nodes=ZP[Coupling['Partition'].loc['ActivePowerDemand']]
ZV=read_ZV(cfg)  # coupling constraints 
TU=read_TU(cfg)  # thermal generation mix
RES=read_RES(cfg)  # renewable generation mix
SS=read_SS(cfg)  # seasonal storage
STS=read_STS(cfg)  # short term storage
IN=read_IN(cfg)  # interconnections

read config from  settings_power_uc_stoch.yml
Coupling constraints:ActivePowerDemand
Emissions constraints:
Read file Data/envis13_stoch/ZP_ZonePartition.csv
Read file Data/envis13_stoch/ZV_ZoneValues.csv
Read file Data/envis13_stoch/TU_ThermalUnits.csv
Read file Data/envis13_stoch/RES_RenewableUnits.csv
Read file Data/envis13_stoch/SS_SeasonalStorage.csv
Read file Data/envis13_stoch/STS_ShortTermStorage.csv
Read file Data/envis13_stoch/IN_Interconnections.csv


In [16]:
# creates the calendar
# beginTS / endTS define the dates of the first and last timestep in the timeseries
# beginDS / end DS define the dates of the first and last timestep of the dataset we are creating
dates=pd.Series()
beginTS=parse_datetime_index(cfg['Calendar']['BeginTimeSeries'],dayfirst=cfg['Calendar']['dayfirst'])
beginDS=parse_datetime_index(cfg['Calendar']['BeginDataset'],dayfirst=cfg['Calendar']['dayfirst'])
endTS=parse_datetime_index(cfg['Calendar']['EndTimeSeries'],dayfirst=cfg['Calendar']['dayfirst'])
endDS=parse_datetime_index(cfg['Calendar']['EndDataset'],dayfirst=cfg['Calendar']['dayfirst'])
UCTimeStep=cfg['Calendar']['TimeStep']['Duration']
UnitUC=cfg['Calendar']['TimeStep']['Unit']
if UnitUC=='days': UCTimeStep=UCTimeStep*24
if UnitUC=='weeks': UCTimeStep=UCTimeStep*168

# the following is used for adapting the timeseries (they are available for a year between jan1 and dec31
# and we may want to create a dataset from eg jun3 yearN to apr6 yearN+2)
dates['UCBeginData']=pd.Timestamp(year=beginTS.year,month=beginTS.month,day=beginTS.day,hour=beginTS.hour,minute=beginTS.minute)
dates['UCBegin']=pd.Timestamp(year=beginDS.year,month=beginDS.month,day=beginDS.day,hour=beginDS.hour,minute=beginDS.minute)
dates['UCEndData']=pd.Timestamp(year=endTS.year,month=endTS.month,day=endTS.day,hour=endTS.hour,minute=endTS.minute)
dates['UCEnd']=pd.Timestamp(year=endDS.year,month=endDS.month,day=endDS.day,hour=endDS.hour,minute=endDS.minute)
dates['UCBeginDataYearP4R']=pd.Timestamp(year=dates['UCBegin'].year,month=dates['UCBeginData'].month,day=dates['UCBeginData'].day,hour=dates['UCBeginData'].hour,minute=dates['UCBeginData'].minute)
DurationTimeSeries=pd.Timedelta(dates['UCEndData']-dates['UCBeginData'])
if dates['UCBegin']<dates['UCBeginDataYearP4R']:
	dates['UCBeginDataYearP4R']=pd.Timestamp(year=dates['UCBegin'].year-1,month=dates['UCBeginData'].month,day=dates['UCBeginData'].day,hour=dates['UCBeginData'].hour,minute=dates['UCBeginData'].minute)
dates['UCEndDataYearP4R']=dates['UCBeginDataYearP4R']+DurationTimeSeries
dates['UCBeginExtendedData']=dates['UCBeginDataYearP4R']
dates['UCEndExtendedData']=dates['UCEndDataYearP4R']
durationInstance=dates.UCEnd-dates.UCBegin+pd.Timedelta(hours=1)
NumberUCTimeSteps=int((durationInstance.days*24+durationInstance.seconds/3600)/UCTimeStep)
durationUCTimeStep=pd.Timedelta(str(UCTimeStep)+' hours')

# create dataframe with start and end dates of all UC timesteps 
datesData=pd.DataFrame(columns=['start','end'])
start=dates['UCBeginData']
i=0
while start<=dates['UCEndData']:
	datesData.loc[i]=[start,start+durationUCTimeStep]
	start=start+durationUCTimeStep
	i=i+1

In [17]:
# read timeseries (for elec demand, inflows, res potentials)
DeterministicTimeSeries=read_deterministic_timeseries(cfg,datesData,dates,UCTimeStep)
ListScenarios=[str(elem) for elem in cfg['Scenarios'] ]
base_scenario='2030_0'   # used for creating the deterministic dataset
DemandScenarios=create_demand_scenarios(cfg,Nodes,Coupling,ZV,ListScenarios,DeterministicTimeSeries,dates,UCTimeStep)
RESScenarios=create_res_scenarios(cfg,RES,ListScenarios,DeterministicTimeSeries,dates,UCTimeStep)
InflowsScenarios=create_inflows_scenarios(cfg,SS,ListScenarios,DeterministicTimeSeries,dates,UCTimeStep)

Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_FR_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_DE_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_IT_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_CH_2030.csv
Read Data/TimeSeries/AggregatedTimeSerie_Total_Eastern Europe.csv
Read Data/TimeSeries/AggregatedTimeSerie_Total_Benelux.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_ES_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_PT_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_IE_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_GB_2030.csv
Read Data/TimeSeries/AggregatedTimeSerie_Total_Balkans.csv
Read Data/TimeSeries/AggregatedTimeSerie_Total_Baltics.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_SE_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_FI_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_DK_2030.csv
Read Data/TimeSeries/P_CMIP6_SP370_EDM_ADM0_NO_2030.csv
Read Data/TimeSeries/AggregatedTimeSerie_Solar-CSP_Balkans.csv
Read Data/TimeSeries/P

In [18]:
# compute coordinates of each bus
MyMap=create_map(cfg,list(Nodes))

create map
use private map


In [37]:
# Create an empty PyPSA network
npypsa = pypsa.Network()
n_snapshots=NumberUCTimeSteps
npypsa.set_snapshots(
        DeterministicTimeSeries['One'].index,
        default_snapshot_weightings=UCTimeStep
    )
carriers=["wind","solar","nuclear","oil","gas","hydrogen","coal","biomass","geothermal","hydro","ocean","battery","AC","NTC","electricity","slack"]
for c in carriers:
    npypsa.add("Carrier",c)

In [38]:
# create one bus per country, with a load (=elec demand) and a slack generator
for node in Nodes:
    x=MyMap.loc[node]['x']
    y=MyMap.loc[node]['y']
    npypsa.add("Bus",node,x=x,y=y,carrier="electricity")
    npypsa.add("Load", "Load_"+node, bus=node, p_set=DemandScenarios[node][base_scenario],
               carrier="electricity")
    npypsa.add("Generator", "Slack_"+node, p_nom=10000, marginal_cost=10000, control="PQ", 
               bus=node, carrier="slack")

In [39]:
# Add the interconnection lines
if 'Efficiency' not in IN.columns:
    IN['Efficiency']=1

for line in IN.index:
    if cfg['Parameters']['extendable']:
        p_nom_extendable=(IN.loc[line]['MaxAddedCapacity']>0)
    else:
        p_nom_extendable=False
    npypsa.add(
            "Link",
            line,
            bus0=IN.loc[line]['StartLine'],
            bus1=IN.loc[line]['EndLine'],
            efficiency=IN.loc[line]['Efficiency'],
            p_nom=IN.loc[line]['MaxPowerFlow'],
            p_max_pu=1,
            p_min_pu=IN.loc[line]['MinPowerFlow']/IN.loc[line]['MaxPowerFlow'],
            p_nom_extendable=p_nom_extendable,
            p_nom_max=IN.loc[line]['MaxPowerFlow']+IN.loc[line]['MaxAddedCapacity'],
            p_nom_min=IN.loc[line]['MaxPowerFlow']-IN.loc[line]['MaxRetCapacity'],
            capital_cost=IN.loc[line]['InvestmentCost']
        )


In [40]:
# create 1 storage unit for each seasonal storage (= hydro reservoirs)
# the inflows can be made stochastic
for (name, zone), row in SS.iterrows():
    if 'InitialVolume' in row:
        state_of_charge_initial=row['InitialVolume']
    elif zone in cfg['Parameters']['InitialFillingrate']:
        state_of_charge_initial=row['MaxVolume']*cfg['Parameters']['InitialFillingrate'][zone]
    else:
        state_of_charge_initial=0
    npypsa.add(
            "StorageUnit",  
            name+"_"+zone,
            control="PQ",
            carrier=cfg['Technos']['SS'][name]['carrier'],
            bus=zone,  
            p_nom=row['MaxPower'],  
            p_min_pu=row['MinPower'],  
            efficiency_store=row['PumpingEfficiency'],
            efficiency_dispatch=row['TurbineEfficiency'],
            max_hours=(row['MaxVolume']/row['MaxPower'])/UCTimeStep,
            inflow=InflowsScenarios.loc[(name,zone)][base_scenario],
            cyclic_state_of_charge=False,
            p_nom_extendable=False,
            capital_cost=1,
            p_nom_max=row['MaxPower'],
            p_nom_min=row['MaxPower'],
            state_of_charge_initial=state_of_charge_initial
        )
    if 'MaxPowerProfile' in SS.columns:
        npypsa.storage_units_t.p_max_pu.loc[:, npypsa.storage_units.bus == zone ]=row['MaxPower']*DeterministicTimeSeries[row['MaxPowerProfile']]
    if 'MinPowerProfile' in SS.columns:
        npypsa.storage_units_t.p_min_pu.loc[:, npypsa.storage_units.bus == zone ]=row['MinPower']*DeterministicTimeSeries[row['MinPowerProfile']]


In [41]:
# create one storageunit for each short term storage (= pumped hydro and batteries)
for (name, zone), row in STS.iterrows():
    if cfg['Parameters']['extendable']:
        p_nom_extendable=(row['MaxAddedCapacity']>0)
    else:
        p_nom_extendable=False
    
    npypsa.add(
            "StorageUnit",  
            name+"_"+zone,
            carrier=cfg['Technos']['STS'][name]['carrier'],
            bus=zone, 
            control="PQ",
            p_nom=row['MaxPower'],  
            p_min_pu=row['MinPower'],  
            capital_cost=max(1,row['CapitalCost']), 
            fom_cost=row['AnnualFixedCost'], 
            lifetime=row['Lifetime'],
            marginal_cost=0,
            efficiency_store=row['PumpingEfficiency'],
            efficiency_dispatch=row['TurbineEfficiency'],
            state_of_charge_initial=0,
            max_hours=(row['MaxVolume']/row['MaxPower'])/UCTimeStep,
            cyclic_state_of_charge=True,
            p_nom_extendable=p_nom_extendable,
            p_nom_max=row['MaxPower']+row['MaxAddedCapacity'],
            p_nom_min=row['MaxPower']+row['MaxRetCapacity'],
        )
    if 'InitialVolume' in STS.columns:
        npypsa.storage_units.at[name+"_"+zone,"state_of_charge_initial"]=row['InitialVolume']
    if 'VolumeLevelTarget' in STS.columns:
        npypsa.storage_units.at[name+"_"+zone,"state_of_charge_initial"]=row['VolumeLevelTarget']
    if 'MaxPowerProfile' in STS.columns:
        npypsa.storage_units_t.p_max_pu.loc[:, name+"_"+zone]=row['MaxPower']*DeterministicTimeSeries[row['MaxPowerProfile']]
    if 'MinPowerProfile' in STS.columns:
        npypsa.storage_units_t.p_min_pu.loc[:, name+"_"+zone ]=row['MinPower']*DeterministicTimeSeries[row['MinPowerProfile']]
    if 'Inflows' in STS.columns and row['Inflows'] > 0:
        if 'InflowsProfile' in STS.columns:
            npypsa.storage_units_t.inflow.loc[:, name+"_"+zone ]=row['Inflows']*DeterministicTimeSeries[row['InflowsProfile']]
        else:
            npypsa.storage_units_t.inflow.loc[:, name+"_"+zone]=row['Inflows']*DeterministicTimeSeries['One']
        

In [42]:
# create one generator for each rebewable generation technology in each country
# potentials can be made stochastic
for (name, zone), row in RES.iterrows():
    if 'Hydro|Run of River' in name:
        p_nom=row['Energy']
    else:
        p_nom=row['MaxPower']
    if p_nom>0:
        if cfg['Parameters']['extendable']:
            p_nom_extendable=(row['MaxAddedCapacity']>0)
        else:
            p_nom_extendable=False
        
        npypsa.add(
                "Generator",  
                name+"_"+zone,
                carrier=cfg['Technos']['RES'][name]['carrier'],
                bus=zone, 
                control="PQ",
                #p_max_pu=(1/p_nom)*RESScenarios.loc[(name,zone)][base_scenario],
                p_max_pu=RESScenarios.loc[(name,zone)][base_scenario],
                p_nom=p_nom,  
                capital_cost=max(0.1,row['CapitalCost']), 
                fom_cost=row['AnnualFixedCost']*UCTimeStep/8760, 
                lifetime=row['Lifetime'],
                marginal_cost=0,
                p_nom_extendable=p_nom_extendable,
                p_nom_max=(row['MaxPower']+row['MaxAddedCapacity']),
                p_nom_min=(row['MaxPower']-row['MaxRetCapacity'])
            )

In [43]:
# create one generator per thermal power plant (or per technology if wished)
for (name, zone), row in TU.iterrows():
    p_nom=row['MaxPower']  # or row['Capacity'] if defining availability timeseries based on average availability in row['AvailabilityRate']
    if p_nom>0:
        if 'MaxPower' in cfg['Technos']['TU'][name] and cfg['Parameters']['committable']:
            p_nom_mod=int(cfg['Technos']['TU'][name]['MaxPower'])
        else:
            p_nom_mod=p_nom
        if cfg['Parameters']['committable']:
            Nb_Full=int(p_nom/p_nom_mod)
            Last=p_nom-p_nom_mod*Nb_Full
            if Last>0: 
                Nb=Nb_Full+1
            else:
                Nb=Nb_Full
        else:
            Nb=1
            Nb_Full=1
            Last=p_nom
        
        for n in range(Nb):
            namen=name+"_"+zone+'_'+str(n)
            if namen not in npypsa.generators.index and zone in npypsa.buses.index:      
                pmax=p_nom_mod
                if n==Nb_Full:
                    pmax=Last
                if zone not in npypsa.buses.index:
                    print(namen,cfg['Technos']['TU'][name]['carrier'],zone,zone in npypsa.buses.index,pmax)
                if cfg['Parameters']['extendable']:
                    p_nom_extendable=(row['MaxAddedCapacity']>0)
                else:
                    p_nom_extendable=False
                if ( ('MinPower' not in cfg['Technos']['TU'][name]) or ('MinPower' in cfg['Technos']['TU'][name] and pmax >= cfg['Technos']['TU'][name]['MinPower'])):
                    npypsa.add(
                            "Generator",  
                            namen,
                            carrier=cfg['Technos']['TU'][name]['carrier'],
                            bus=zone, 
                            control="PQ",
                            p_nom=pmax,
                            capital_cost=row['CapitalCost'], 
                            fom_cost=row['AnnualFixedCost']*UCTimeStep/8760, 
                            lifetime=row['Lifetime'],
                            marginal_cost=row['VariableCost'], 
                            p_nom_extendable=p_nom_extendable,
                            p_nom_max=p_nom+row['MaxAddedCapacity']*row['AvailabilityRate'],
                            p_nom_min=p_nom -row['MaxRetCapacity']*row['AvailabilityRate'],
                            committable=cfg['Parameters']['committable'],
                            p_init=pmax,
                            up_time_before=1000
                    )
                    if cfg['Parameters']['committable']:
                        if 'MinOnDuration' in cfg['Technos']['TU'][name]:
                            npypsa.generators.at[namen,'min_up_time'] = min(math.ceil(cfg['Technos']['TU'][name]['MinOnDuration']/UCTimeStep),len(npypsa.snapshots)*UCTimeStep)
                        if 'MinOffDuration' in cfg['Technos']['TU'][name]:
                            npypsa.generators.at[namen, 'min_down_time'] = min(math.ceil(cfg['Technos']['TU'][name]['MinOffDuration']/UCTimeStep),len(npypsa.snapshots)*UCTimeStep)
                        if 'MinPower' in cfg['Technos']['TU'][name]:
                            npypsa.generators.at[namen, 'p_min_pu'] = cfg['Technos']['TU'][name]['MinPower']/pmax
                        if 'RampRate' in cfg['Technos']['TU'][name]:
                            npypsa.generators.at[namen, 'ramp_limit_up']= p_nom_mod*cfg['Technos']['TU'][name]['RampRate']/100
                        if 'RampRate' in cfg['Technos']['TU'][name]:
                            npypsa.generators.at[namen, 'ramp_limit_down']= p_nom_mod*cfg['Technos']['TU'][name]['RampRate']/100
                        if 'MinPower' in cfg['Technos']['TU'][name] and pmax < cfg['Technos']['TU'][name]['MinPower']:
                            print('remove ', namen, ' pmin>pmax')
                            npypsa.remove("Generator",namen)
                        if 'StartUpCost' in cfg['Technos']['TU'][name]:
                            npypsa.generators.at[namen, 'start_up_cost']= cfg['Technos']['TU'][name]['StartUpCost']*pmax/p_nom_mod


In [44]:
npypsa.export_to_netcdf("pypsaNetwork.nc")
npypsa.sanitize()

INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'pypsaNetwork.nc contains: carriers, buses, generators, links, storage_units, loads
INFO:pypsa.consistency:Sanitizing network...
INFO:pypsa.components._types.carriers:Assigned colors to 16 carriers using 'tab10' palette.
INFO:pypsa.consistency:Network sanitization complete.


In [45]:
# sumplify network by keepiing only a selection of regions
Buses_Reduced=['France']
n_reduced=npypsa.copy().slice_network(buses=Buses_Reduced)

In [46]:
ndet=npypsa.copy()
cplex_solver_options= {
    "lpmethod": 4,
    "timelimit": 600,
}
ndet.optimize(solver_name="cplex", solver_options=cplex_solver_options)

/tmp/ipykernel_4895/1522851641.py:6: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  ndet.optimize(solver_name="cplex", solver_options=cplex_solver_options)
INFO:linopy.model: Solve problem using Cplex solver
INFO:linopy.model:Solver options:
 - lpmethod: 4
 - timelimit: 600
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 242.06it/s]
INFO:linopy.io: Writing time: 0.26s


Version identifier: 22.1.2.0 | 2024-11-26 | 0edbb82fd
CPXPARAM_Read_DataCheck                          1
CPXPARAM_LPMethod                                4
CPXPARAM_TimeLimit                               600
Tried aggregator 1 time.
LP Presolve eliminated 17160 rows and 2619 columns.
Aggregator did 216 substitutions.
Reduced LP has 2600 rows, 6614 columns, and 12744 nonzeros.
Presolve time = 0.04 sec. (14.74 ticks)
Parallel mode: using up to 4 threads for barrier.

***NOTE: Found 17 dense columns.

Number of nonzeros in lower triangle of A*A' = 6775
Using Approximate Minimum Degree ordering
Total time for automatic ordering = 0.02 sec. (1.18 ticks)
Summary statistics for Cholesky factor:
  Threads                   = 4
  Rows in Factor            = 2617
  Integer space required    = 8465
  Total non-zeros in factor = 50594
  Total FP ops to factor    = 2403672
 Itn      Primal Obj        Dual Obj  Prim Inf Upper Inf  Dual Inf Inf Ratio
   0   2.1115527e+11  -5.4657318e+13  1.37e+07  2

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 9449 primals, 19976 duals
Objective: 1.10e+08
Solver model: available
Solver message: optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [65]:
# make network stochastic
# does not work with integer constraints

# ListScenarios contains the whole list of available scenarios
# for a simulation on 2030, it is possible to use all scenarios between 2020_0 and 2040_4

# Scenarios=ListScenarios   # if you wish to include all (105) scenarios
# Below we reduce the list for creating a smaller dataset
Scenarios=["2030_0","2030_1","2030_3"]
for sc in Scenarios:
    if sc not in ListScenarios:
        print('scenario ',sc,' not in dataset ')
NumberScenarios = len(Scenarios)
Probability=1/NumberScenarios
Probas={ scenario : Probability for scenario in Scenarios }

nsto=npypsa.copy()
nsto.set_scenarios(Probas)
for sc in Scenarios:
    # no stochastic inflow yet in pypsa2smspp
    #for (name, zone), row in SS.iterrows():
        # add stochastic inflows to "seasonal storage' storageunits
    #    nsto.storage_units_t.inflow.loc[:, (sc,name+"_"+zone)]=InflowsScenarios.loc[(name,zone)][sc]
    for (name, zone), row in RES.iterrows():
        if row['MaxPower']>0:
            # add stochastic maxpower to renewable generation
            nsto.generators_t.p_max_pu.loc[:, (sc,name+"_"+zone)]=RESScenarios.loc[(name,zone)][sc]   
load_timeseries=pd.DataFrame( 
    index=DeterministicTimeSeries['One'].index,
    columns= pd.MultiIndex.from_product(
        [nsto.scenarios, ["Load_"+node for node in Nodes ] ], 
        names =["scenario","name"]
    ),
)
for node in Nodes:
    for scen in Scenarios:
        load_timeseries.loc[:, (scen, "Load_"+node )] = DemandScenarios[node][scen]
nsto.c.loads.dynamic.p_set = load_timeseries

In [66]:
nsto_reduced=n_reduced.copy()

nsto_reduced.set_scenarios(Probas)
for sc in Scenarios:
    for (name, zone), row in SS.iterrows():
        if zone in nsto_reduced.buses.index:
            nsto_reduced.storage_units_t.inflow.loc[:, (sc,name+"_"+zone)]=InflowsScenarios.loc[(name,zone)][sc]
    for (name, zone), row in RES.iterrows():
        if zone in nsto_reduced.buses.index:
            if row['MaxPower']>0:
                nsto_reduced.generators_t.p_max_pu.loc[:, (sc,name+"_"+zone)]=RESScenarios.loc[(name,zone)][sc]   
load_TS_reduced=[ 'Load_'+region for region in Buses_Reduced ]
load_timeseries=pd.DataFrame( 
    index=DeterministicTimeSeries['One'].index,
    columns= pd.MultiIndex.from_product(
        [nsto_reduced.scenarios, load_TS_reduced ], 
        names =["scenario","name"]
    ),
)
for node in Buses_Reduced:
    for scen in Scenarios:
        load_timeseries.loc[:, (scen, "Load_"+node )] = DemandScenarios[node][scen]
nsto_reduced.c.loads.dynamic.p_set = load_timeseries

In [67]:
nsto_pypsa=nsto.copy()
nsto_pypsa.optimize(solver_name="cplex", solver_options=cplex_solver_options)

/tmp/ipykernel_4895/4275004583.py:2: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  nsto_pypsa.optimize(solver_name="cplex", solver_options=cplex_solver_options)
INFO:linopy.model: Solve problem using Cplex solver
INFO:linopy.model:Solver options:
 - lpmethod: 4
 - timelimit: 600
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 228.35it/s]
INFO:linopy.io: Writing time: 0.25s


Version identifier: 22.1.2.0 | 2024-11-26 | 0edbb82fd
CPXPARAM_Read_DataCheck                          1
CPXPARAM_LPMethod                                4
CPXPARAM_TimeLimit                               600
Tried aggregator 1 time.
LP Presolve eliminated 51480 rows and 7789 columns.
Aggregator did 648 substitutions.
Reduced LP has 7800 rows, 19782 columns, and 38232 nonzeros.
Presolve time = 0.08 sec. (44.27 ticks)
Parallel mode: using up to 4 threads for barrier.

***NOTE: Found 30 dense columns.

Number of nonzeros in lower triangle of A*A' = 14856
Using Approximate Minimum Degree ordering
Total time for automatic ordering = 0.01 sec. (2.55 ticks)
Summary statistics for Cholesky factor:
  Threads                   = 4
  Rows in Factor            = 7830
  Integer space required    = 25793
  Total non-zeros in factor = 132402
  Total FP ops to factor    = 5748340
 Itn      Primal Obj        Dual Obj  Prim Inf Upper Inf  Dual Inf Inf Ratio
   0   2.1493772e+11  -4.0993003e+13  5.07e+0

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 28219 primals, 59928 duals
Objective: 1.11e+08
Solver model: available
Solver message: optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Link-ext-p-lower, Link-ext-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [32]:
# transform deterministic network to sms++ and solve with ucblock_solver
n_clean_det=npypsa.copy()

myconfigfile="/home/F04340/RESILIENT/SMSpp_PyPSA/MyNotebook/settings/uc_solverconfig.txt"
temporary_smspp_file=cfg["outputDir"]+"temp_uc_network.nc"
output_file=cfg["outputDir"]+"temp_output_uc_file.txt"
mysolution="sms_uc_solution.nc"
#mysolver_path="/home/F04340/SMSPP/smspp-project/bin/ucblock_solver"
mysolver_path="/home/F04340/P4R3/p4r-env/bin/p4r ucblock_solver"
kwargs={"t":"0"}

tran=pypsa2smspp.Transformation( fp_solution=mysolution,configfile=myconfigfile,
                                 pysmspp_options={"solver_path":mysolver_path,"shell":True,**kwargs} )

#sms_network= tran.run( nsms,verbose=True) # convert the network to be solved as UCBlock
sms_network= tran.create_model( n_clean_det,verbose=True) # convert the network to be solved as UCBlock
tran.optimize(verbose=True)
n_sms_det_solved=tran.retrieve_solution(n_clean_det,verbose=True)

[START] consistency_check
[ OK  ] consistency_check (0.000s)
[START] direct
[ OK  ] direct (2.244s)
[START] prepare_tssb_interface
[ OK  ] prepare_tssb_interface (0.000s)
[START] convert_to_blocks
[ OK  ] convert_to_blocks (0.864s)
[START] optimize
Executing command:
/home/F04340/P4R3/p4r-env/bin/p4r ucblock_solver temp_test_case.nc -S uc_solverconfig.txt -c /home/F04340/RESILIENT/SMSpp_PyPSA/MyNotebook/settings/ -p /home/F04340/RESILIENT/SMSpp_PyPSA/MyNotebook/output/ -O /home/F04340/RESILIENT/SMSpp_PyPSA/MyNotebook/output/sms_uc_solution.nc -t 0

Using a default Block configuration
Solver: CPXMILPSolver
Elapsed time: 2.21782589e-01 s
Status = 10 (Success)
Upper bound = 1.81957143e+10
Lower bound = 1.81957143e+10
Peak CPU Usage: 0.00 %
Peak Memory Usage: 0.90 MB
Total Time: 9.27 seconds

[ OK  ] optimize (11.296s)
[START] parse_solution_to_unitblocks
[split] No merged DCNetworkBlock entries found; nothing to do.
[ OK  ] parse_solution_to_unitblocks (0.005s)
[START] inverse_transformat

In [33]:
pypsa_stats=ndet.statistics.energy_balance(comps=["Generator"]).droplevel([0,2])
smspp_stats=n_sms_det_solved.statistics.energy_balance(comps=["Generator"]).droplevel([0,2])
diff_stats=pypsa_stats-smspp_stats
merged_stats=pd.concat([pypsa_stats,smspp_stats,diff_stats],axis=1)
merged_stats.columns=["pypsa","smspp","difference"]
print(merged_stats)
pypsa_obj = ndet.statistics.opex().sum()
smspp_obj = n_sms_det_solved.statistics.opex().sum()
print("SMS++ obj         : %.6f" % smspp_obj)
print("PyPSA dispatch obj: %.6f" % pypsa_obj)
print("Error SMS++ - PyPSA dispatch [%%]: %.5f" % (100*(smspp_obj - pypsa_obj)/pypsa_obj))

                   pypsa         smspp    difference
carrier                                             
biomass     1.685511e+04  1.810054e+04   -1245.43531
coal        1.363398e+06  1.387379e+06  -23981.54067
gas         1.994689e+06  1.768921e+06  225767.66232
geothermal  3.319920e+04  3.319920e+04       0.00000
hydro       5.659361e+05  6.278780e+05  -61941.94214
nuclear     1.788991e+06  1.834002e+06  -45010.77235
ocean       3.404304e+03  3.404304e+03       0.00000
solar       2.395997e+06  2.398884e+06   -2886.88617
wind        1.316457e+06  1.322521e+06   -6064.05580
SMS++ obj         : 100592267.542400
PyPSA dispatch obj: 110259661.222240
Error SMS++ - PyPSA dispatch [%]: -8.76784


In [ ]:
# transfor to sms++ and solve with tssb_solver expansion problem
n_clean_sto=nsto.copy()

myconfigfile="/home/F04340/RESILIENT/SMSpp_PyPSA/MyNotebook/settings/TSSBSCfg.txt"
temporary_smspp_file=cfg["outputDir"]+"temp_sto_network.nc"
output_file=cfg["outputDir"]+"temp_sto_output_file.txt"
mysolution="sms_sto_solution.nc"
mysolver_path="/home/F04340/SMSPP/smspp-project/bin/tssb_solver"
#mysolver_path="/home/F04340/P4R3/p4r-env/bin/p4r tssb_solver"
kwargs={}

tran=pypsa2smspp.Transformation( fp_solution=mysolution,
                                 configfile=myconfigfile,
                                capacity_expansion_ucblock=True,
                                 workdir=cfg["outputDir"],
                                 stochastic_parameters={
                                        "stochastic_type": "tssb",
                                        "parameters": ["demand","renewable_maxpower"],
                                    },
                                 pysmspp_options={"solver_path":mysolver_path,"shell":True,**kwargs} )


#sms_sto_network= tran.run( n_clean_sto,verbose=True) # convert the network to be solved as UCBlock

# does not work

sms_sto_network= tran.create_model( n_clean_sto,verbose=True) # convert the network to be solved as UCBlock
tran.optimize(verbose=True)
n_sms_sto_solved=tran.retrieve_solution(n_clean_sto,verbose=True)


